<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/05_phase3_ablations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Phase 3 ablations

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation

Three ablation studies and a final model. Each ablation varies one component of the Phase 2 multimodal model and keeps everything else fixed.

1. **Text encoder:** CLIP vs RemoteCLIP (domain-adapted on remote-sensing pairs).
2. **Loss function:** weighted CE vs CE+Dice vs CE+Lovász.
3. **Fusion mechanism:** FiLM vs pooled cross-attention vs multi-token cross-attention.

The final model combines the two strongest single-component improvements (CE+Lovász + RemoteCLIP) and tests whether their gains stack.

Phase 2 numbers used as reference points: vision-only UNetFormer baseline at test mIoU 0.6488 (`03_baseline_models.ipynb`), and Phase 2 multimodal model at test mIoU 0.6970 (`04_multimodal.ipynb`, CLIP + `text_qwen3-4b` + pooled cross-attention + weighted CE). Both are loaded from saved JSON files.

All experiments use pixel-level mIoU and the Phase 2 training recipe (30 epochs, AdamW lr=6e-4, weight decay 0.01, batch size 8, weighted CE + 0.4 × auxiliary loss, seed 42).

Sections:
1. Setup
2. Dataset and DataLoaders
3. UNetFormer architecture
4. Multimodal architecture
5. Training infrastructure
6. Ablation 1 — Text encoder (RemoteCLIP)
7. Ablation 2 — Loss function (Dice, Lovász)
8. Ablation 3 — Fusion mechanism (FiLM, multi-token)
9. Final model — CE+Lovász + RemoteCLIP
10. Unified results and qualitative figure


## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies. open_clip_torch provides the frozen CLIP text encoder.
# huggingface_hub is needed to download RemoteCLIP weights for Ablation 1.
!pip install transformers timm einops wandb open_clip_torch huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00


In [3]:
# Project paths (Drive) and a local working copy for fast I/O during training
from pathlib import Path
import shutil

PROJECT_DIR      = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE   = PROJECT_DIR / 'data'
SPLITS_CSV       = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR  = PROJECT_DIR / 'checkpoints'
RESULTS_DIR      = PROJECT_DIR / 'results'
REMOTECLIP_CACHE = PROJECT_DIR / 'remoteclip_cache'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
REMOTECLIP_CACHE.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA  = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [5]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time, json, random, textwrap
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

import wandb
wandb.login()
WANDB_PROJECT = 'di725_project'

Device: cuda


In [6]:
# Class definitions (same across all project notebooks)
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# Canonical class palette from notebook 01, used by the qualitative figure in Section 10.
CLASS_RGB = {
    'Tree':     (0,   100, 0),
    'Shrub':    (255, 182, 193),
    'Grass':    (154, 205, 50),
    'Crop':     (255, 215, 0),
    'Built-up': (139, 69,  19),
    'Barren':   (211, 211, 211),
    'Water':    (0,   0,   255),
}

# 5 caption variants from the dataset
CAPTION_COLS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

# Best caption variant from Phase 2
BEST_CAPTION = 'text_qwen3-4b'

# Load split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

# Class weights via median frequency balancing (computed from training split)
class_avg = train_df[CLASS_NAMES].mean().values
class_freq = class_avg / class_avg.sum()
median_freq = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights.round(2)}')

Train: 7000  |  Val: 1500  |  Test: 1500
Class weights: [0.15 5.12 0.09 0.24 3.77 1.   2.54]


In [7]:
# Phase 2 reference numbers, loaded from saved JSON files
with open(RESULTS_DIR / 'baselines_results.json') as f:
    baseline_data = json.load(f)

with open(RESULTS_DIR / 'multimodal_results.json') as f:
    multimodal_data = json.load(f)

# Vision-only UNetFormer baseline (Phase 2)
BASELINE_MIOU = baseline_data['unetformer']['test_miou']
BASELINE_OA   = baseline_data['unetformer']['test_oa']
BASELINE_IOUS = baseline_data['unetformer']['class_ious']

# Phase 2 multimodal model: CLIP + pooled cross-attention + weighted CE on text_qwen3-4b.
# Used as the reference point for Phase 3 comparisons.
PHASE2_MIOU = multimodal_data['multimodal'][BEST_CAPTION]['test_miou']
PHASE2_OA   = multimodal_data['multimodal'][BEST_CAPTION]['test_oa']
PHASE2_IOUS = multimodal_data['multimodal'][BEST_CAPTION]['class_ious']

print(f'Vision-only baseline: test mIoU {BASELINE_MIOU:.4f}')
print(f'Phase 2 multimodal  ({BEST_CAPTION}): test mIoU {PHASE2_MIOU:.4f}')

Vision-only baseline: test mIoU 0.6488
Phase 2 multimodal  (text_qwen3-4b): test mIoU 0.6970


## 2. Dataset and DataLoaders

Returns (image, mask, caption) tuples. The caption column is a constructor argument so the same class works for any of the 5 caption variants. Same as `04_multimodal.ipynb`.

In [8]:
# Dataset class — reads pre-converted class-index masks and a caption column
class RSMultiModalDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])
        return img, mask, caption

In [9]:
# DataLoader factory — train/val/test loaders for any caption column
BATCH_SIZE  = 8
NUM_WORKERS = 4

def make_loaders(caption_col):
    """Build train/val/test loaders for a given caption column."""
    train_dataset = RSMultiModalDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

# Sanity check with the best caption variant from Phase 2
train_loader, val_loader, test_loader = make_loaders(BEST_CAPTION)
print(f'Caption column: {BEST_CAPTION}')
print(f'Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')

imgs, masks, captions = next(iter(train_loader))
print(f'Batch: images {imgs.shape}, masks {masks.shape}')

Caption column: text_qwen3-4b
Train: 7000 samples, 875 batches
Batch: images torch.Size([8, 3, 256, 256]), masks torch.Size([8, 256, 256])


## 3. UNetFormer architecture

Same UNetFormer as in `03_baseline_models.ipynb` (CNN encoder `swsl_resnet18` with Global-Local Transformer Block decoder, Feature Refinement Head, and auxiliary head for deep supervision). Re-defined here so this notebook is self-contained.

Source: https://github.com/WangLibo1995/GeoSeg

In [10]:
# Building blocks (Conv + BN + ReLU variants)
from einops import rearrange
from timm.models.layers import DropPath, trunc_normal_
import timm


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [11]:
# MLP and Global-Local Attention (window self-attention + local conv branch)
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x); return x


class GlobalLocalAttention(nn.Module):
    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                    padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                    padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws); coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1; relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0: x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0: x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws); B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1); attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local; out = self.pad_out(out); out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [12]:
# Transformer Block (GLTB): pre-norm, attention, residual, MLP, residual
class Block(nn.Module):
    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [13]:
# Decoder components: weighted feature fusion (WF), feature refinement head, aux head
class WF(nn.Module):
    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x); shortcut = self.shortcut(x)
        pa = self.pa(x) * x; ca = self.ca(x) * x
        x = pa + ca; x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x); feat = self.drop(feat); feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

## 4. Multimodal architecture

Three additions to the vision-only UNetFormer:

1. **Frozen CLIP text encoder** (`ViT-B-32`, `laion2b_s34b_b79k`): converts each caption into a 512-d L2-normalized embedding. No gradients flow into CLIP.
2. **Text-Visual Cross-Attention**: visual features attend to the text embedding via a gated residual. The gate starts at zero, so text influence ramps up gradually during training instead of disrupting the pretrained backbone.
3. **TextAugmentedDecoder**: same as the UNetFormer decoder, with a cross-attention module after each Global-Local Transformer Block and after the Feature Refinement Head.

Only the cross-attention modules and the decoder are trained. The CNN encoder (`swsl_resnet18`) and CLIP text encoder are both frozen.

In [14]:
import open_clip


class CLIPTextEncoder(nn.Module):
    """Frozen CLIP text encoder. Outputs L2-normalized 512-d embeddings."""

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        # Drop the visual tower since only the text encoder is used (saves ~88M frozen params)
        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        # Freeze all remaining CLIP parameters
        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features


# Verify
clip_enc = CLIPTextEncoder().to(device)
with torch.no_grad():
    emb = clip_enc(['test caption'])
    print(f'CLIP text encoder loaded: output {emb.shape}')
    print(f'Frozen params: {sum(p.numel() for p in clip_enc.parameters()) / 1e6:.1f}M')
del clip_enc
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP text encoder loaded: output torch.Size([1, 512])
Frozen params: 63.4M


In [15]:
class TextVisualCrossAttention(nn.Module):
    """Visual features attend to text embedding via gated residual.

    The gate starts at 0, so text initially has no influence. As training
    progresses, the gate learns to weight the text contribution.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text embedding into the visual feature space
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )
        # Gated residual — starts at 0, learns to scale text contribution
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        text_feat = self.text_proj(text_embed)
        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        v = self.v_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [16]:
class TextAugmentedDecoder(nn.Module):
    """UNetFormer decoder + cross-attention after each decoder stage."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # Text-visual cross-attention at each decoder stage
        self.text_attn4 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn3 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn2 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn1 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed); h2 = x
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed)
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [17]:
class UNetFormerCLIP(nn.Module):
    """UNetFormer + frozen CLIP text encoder + cross-attention fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7,
                 clip_model='ViT-B-32', clip_pretrained='laion2b_s34b_b79k'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder(clip_model, clip_pretrained)
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)


# Sanity check: instantiate, count parameters, forward pass in both modes
test_model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Total parameters    : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')
print(f'Frozen parameters   : {frozen_params / 1e6:.2f}M (CLIP text encoder)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])

    test_model.eval()
    out_eval = test_model(sample_imgs, sample_caps)
    print(f'\nEval  mode: input {sample_imgs.shape} -> output {out_eval.shape}')

    test_model.train()
    out_main, out_aux = test_model(sample_imgs, sample_caps)
    print(f'Train mode: main {out_main.shape}, aux {out_aux.shape}')

del test_model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Total parameters    : 75.37M
Trainable parameters: 11.94M
Frozen parameters   : 63.43M (CLIP text encoder)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Eval  mode: input torch.Size([2, 3, 256, 256]) -> output torch.Size([2, 7, 256, 256])
Train mode: main torch.Size([2, 7, 256, 256]), aux torch.Size([2, 7, 256, 256])


## 5. Training infrastructure

Same training recipe as Phase 2: 30 epochs, AdamW (lr=6e-4, weight decay 0.01), CosineAnnealingLR, weighted CE + 0.4 × auxiliary head loss, seed 42.

The `train_multimodal` function below accepts the model and optional loss criterion as arguments. The model defaults to `UNetFormerCLIP` and the criterion defaults to weighted cross-entropy. Sections 6-9 pass different models (RemoteCLIP, FiLM, multi-token variants) and different losses (CE+Dice, CE+Lovász) without modifying the training loop.

Each run is logged to Weights & Biases under the project `di725_project`.

In [18]:
# Training hyperparameters (same as Phase 2)
NUM_EPOCHS      = 30
LR              = 6e-4
WEIGHT_DECAY    = 0.01
AUX_LOSS_WEIGHT = 0.4

In [19]:
# Pixel-level IoU helpers — accumulate intersection/union across the full dataset
def update_confusion(intersection, union, pred, target, num_classes=NUM_CLASSES):
    """Accumulate per-class intersection and union counts in-place.
    pred, target: tensors of shape (B, H, W) with class indices.
    Computation stays on GPU; only scalars transfer to CPU via .item()."""
    for c in range(num_classes):
        pred_c   = (pred == c)
        target_c = (target == c)
        intersection[c] += (pred_c & target_c).sum().item()
        union[c]        += (pred_c | target_c).sum().item()


def finalize_iou(intersection, union, num_classes=NUM_CLASSES):
    """Convert accumulated counts into per-class IoU and mIoU.
    Classes absent from both pred and target (union=0) are NaN and excluded from mIoU."""
    class_ious = []
    for c in range(num_classes):
        if union[c] == 0:
            class_ious.append(float('nan'))
        else:
            class_ious.append(intersection[c] / union[c])
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    return class_ious, miou

In [20]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """One training epoch over the (image, mask, caption) loader.

    The criterion must take (logits, target_mask) and return a scalar loss.
    Used for both weighted CE (Sections 6, 8) and composite losses like CE+Dice,
    CE+Lovász (Section 7), via callable criterion objects defined in those sections.
    """
    model.train()
    total_loss = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits_main, logits_aux = model(imgs, list(captions))
        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns: per-class pixel IoU list, pixel-level mIoU, overall accuracy, average loss."""
    model.eval()
    intersection = np.zeros(num_classes, dtype=np.int64)
    union        = np.zeros(num_classes, dtype=np.int64)
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        update_confusion(intersection, union, preds, masks, num_classes)
        total_correct += (preds == masks).sum().item()
        total_pixels  += masks.numel()
    class_ious, miou = finalize_iou(intersection, union, num_classes)
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [21]:
def train_multimodal(caption_col, model_factory=None, criterion=None,
                     num_epochs=NUM_EPOCHS, lr=LR, run_name=None, wandb_config=None):
    """Train a multimodal model and return (history, best_val_miou, checkpoint_path).

    Args:
        caption_col: caption column from CAPTION_COLS to use as text input.
        model_factory: callable returning a fresh model. Defaults to the Phase 2
            multimodal model (CLIP + pooled cross-attention).
        criterion: loss function (logits, target) -> scalar. Defaults to weighted CE.
        run_name: identifier for the W&B run and checkpoint filename.
            Defaults to f'multimodal_{caption_col}'.
        wandb_config: dict added to the W&B run config. Augments the default config
            with experiment-specific fields (loss type, fusion type, encoder, etc.).
    """
    # Defaults — Phase 2 configuration if no overrides are given
    if model_factory is None:
        model_factory = lambda: UNetFormerCLIP(num_classes=NUM_CLASSES)
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    if run_name is None:
        base_name = caption_col.replace('-', '_').replace('/', '_')
        run_name = f'multimodal_{base_name}'

    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    # Loaders and model
    train_loader, val_loader, _ = make_loaders(caption_col)
    model = model_factory().to(device)

    # Optimizer over only trainable params (skips frozen CLIP)
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    # W&B run
    default_config = {
        'caption_col':     caption_col,
        'num_epochs':      num_epochs,
        'lr':              lr,
        'weight_decay':    WEIGHT_DECAY,
        'batch_size':      BATCH_SIZE,
        'aux_loss_weight': AUX_LOSS_WEIGHT,
        'seed':            SEED,
    }
    if wandb_config:
        default_config.update(wandb_config)

    wandb.init(project=WANDB_PROJECT, name=run_name, config=default_config, reinit=True)

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_class_ious, val_miou, val_oa, val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_miou
    wandb.finish()

    # Free the training model from GPU memory before the next experiment
    del model
    torch.cuda.empty_cache()

    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou, save_path

In [22]:
def save_history(history, name):
    """Persist training history to disk so it survives runtime restarts."""
    path = RESULTS_DIR / f'{name}_history.json'
    with open(path, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {path}')


def evaluate_checkpoint(caption_col, ckpt_path, model_factory=None, criterion=None):
    """Reload a trained checkpoint and evaluate on the test set.

    Used by Sections 6-9 to produce the test mIoU after each ablation finishes training.
    """
    if model_factory is None:
        model_factory = lambda: UNetFormerCLIP(num_classes=NUM_CLASSES)
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    _, _, test_loader = make_loaders(caption_col)
    model = model_factory().to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    class_ious, miou, oa, loss = evaluate(model, test_loader, criterion, device)
    del model
    torch.cuda.empty_cache()
    return class_ious, miou, oa, loss

## 6. Ablation 1 — Text encoder pretraining

Tests whether replacing CLIP with RemoteCLIP improves segmentation. RemoteCLIP (Liu et al., 2023) is a CLIP variant fine-tuned on remote-sensing image-text pairs from datasets such as RSITMD, RSICD, and Sydney-Captions. The architecture (`ViT-B-32`) is identical to Phase 2 CLIP, so only the encoder weights change.

The same 5 caption variants from Phase 2 are tested. Phase 2's CLIP results are loaded from `multimodal_results.json` as the reference. Phase 3 trains 5 new models with RemoteCLIP and compares paired test mIoU values.

Reference: Liu et al., "RemoteCLIP: A Vision Language Foundation Model for Remote Sensing." IEEE TGRS 2023.

In [24]:
from huggingface_hub import hf_hub_download


class CLIPTextEncoderRC(nn.Module):
    """Frozen text encoder that supports either CLIP or RemoteCLIP weights.

    Both use ViT-B-32 architecture; only the trained weights differ. RemoteCLIP
    weights are downloaded from HuggingFace (chendelong/RemoteCLIP) and loaded
    into an open_clip ViT-B-32 model with no pretrained weights.
    """

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        if pretrained == 'remoteclip':
            # Create an empty ViT-B-32, then load RemoteCLIP weights into it
            clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=None)
            ckpt_path = hf_hub_download(
                'chendelong/RemoteCLIP',
                f'RemoteCLIP-{model_name}.pt',
                cache_dir=str(REMOTECLIP_CACHE),
            )
            ckpt = torch.load(ckpt_path, map_location='cpu')
            msg = clip_model.load_state_dict(ckpt)
            print(f'RemoteCLIP {model_name} loaded: {msg}')
        else:
            clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)

        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        return text_features / text_features.norm(dim=-1, keepdim=True)


class UNetFormerRemoteCLIP(nn.Module):
    """UNetFormer + frozen RemoteCLIP text encoder + cross-attention fusion.

    Architecture is identical to UNetFormerCLIP except for the text encoder
    weights. Only the text encoder is swapped.
    """

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoderRC(pretrained='remoteclip')
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)


# Sanity check
test_model = UNetFormerRemoteCLIP(num_classes=NUM_CLASSES).to(device)
total = sum(p.numel() for p in test_model.parameters()) / 1e6
trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad) / 1e6
print(f'Total parameters    : {total:.2f}M')
print(f'Trainable parameters: {trainable:.2f}M (same as CLIP-based model)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])
    test_model.eval()
    out = test_model(sample_imgs, sample_caps)
    print(f'Forward pass: {sample_imgs.shape} -> {out.shape}')

del test_model
torch.cuda.empty_cache()

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>
Total parameters    : 75.37M
Trainable parameters: 11.94M (same as CLIP-based model)
Forward pass: torch.Size([2, 3, 256, 256]) -> torch.Size([2, 7, 256, 256])


In [ ]:
# Train UNetFormerRemoteCLIP on all 5 caption variants
remoteclip_factory = lambda: UNetFormerRemoteCLIP(num_classes=NUM_CLASSES)
remoteclip_runs = {}

for caption_col in CAPTION_COLS:
    base_name = caption_col.replace('-', '_').replace('/', '_')
    run_name = f'remoteclip_{base_name}'

    print(f'\n{"=" * 60}')
    print(f'Caption: {caption_col}')
    print(f'{"=" * 60}\n')

    hist, val_miou, ckpt = train_multimodal(
        caption_col=caption_col,
        model_factory=remoteclip_factory,
        run_name=run_name,
        wandb_config={'text_encoder': 'remoteclip', 'experiment': 'ablation_1_encoder'},
    )
    save_history(hist, run_name)
    remoteclip_runs[caption_col] = {'history': hist, 'val_miou': val_miou, 'ckpt': ckpt}


Caption: hybrid_gemma3-4b



RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training remoteclip_hybrid_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.0938   0.4451   0.5119   0.7517   60.0s   BEST
    2   0.7262   0.4610   0.5186   0.7280   58.6s   BEST
    3   0.6110   0.5009   0.5145   0.7718   58.1s       
    4   0.5591   0.3928   0.5136   0.7668   57.7s       
    5   0.5456   0.4522   0.5269   0.7517   58.8s   BEST
    6   0.5038   0.3154   0.5529   0.7977   61.5s   BEST
    7   0.4752   0.3503   0.5626   0.7934   59.0s   BEST
    8   0.4463   0.3755   0.6196   0.8191   58.7s   BEST
    9   0.4665   0.3010   0.5637   0.8137   58.0s       
   10   0.4159   0.3157   0.5599   0.7933   57.3s       
   11   0.3975   0.3093   0.6139   0.8262   57.5s       
   12   0.4013   0.2524   0.6473   0.8483   59.1s   BEST
   13   0.3740   0.2546   0.6388   0.8487   58.0s       
   14   0.3572   0.2491   0.6193   0.8338   58.2s       
   15   0.3515   0.2446   0.6406   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▁▆▂▁▄▄▆▄▃▄▆▇▄▅▆▇▆▆▇▆▇█▇▇▇▇▇██
val_iou/Built-up,▁▅▆▆▅▃▄█▄▃▆▇▅▆▇▆▇▆▇▆▇███▇██▇█▇
val_iou/Crop,▂▁▃▅▃▄▄▅▆▄▅▆▆▇▇▇▆▇▆▇▇▇▇███████
val_iou/Grass,▂▁▄▃▃▄▅▅▅▄▆▆▆▆▆▇▇▇▇▇▇███▇█████
val_iou/Shrub,▃▂▁▂▃▃▃▂▄▄▄▄▄▄▃▄▅▅▅▅▅▆▇▆▇▆▇███
val_iou/Tree,▃▃▂▃▂▄▁▅▅▄▄▇▆▆▆▅▆▇▇▇▇█████████
val_iou/Water,▆▆▁▂▅▅▆▇▅▇█▇█▇████████████████
+3,...



Best validation mIoU: 0.7040
Saved history to /content/drive/MyDrive/DI725_Project/results/remoteclip_hybrid_gemma3_4b_history.json

Caption: hybrid_qwen3-vl-8b



/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


Training remoteclip_hybrid_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1141   0.5046   0.5009   0.7633   59.3s   BEST
    2   0.7107   0.4570   0.4893   0.7493   58.9s       
    3   0.6439   0.3678   0.5546   0.7880   59.3s   BEST
    4   0.5983   0.4103   0.5058   0.7557   57.7s       


In [ ]:
# Test set evaluation for all 5 RemoteCLIP checkpoints
remoteclip_test = {}

for caption_col, run_info in remoteclip_runs.items():
    print(f'Evaluating RemoteCLIP on {caption_col}...')
    class_ious, miou, oa, loss = evaluate_checkpoint(
        caption_col=caption_col,
        ckpt_path=run_info['ckpt'],
        model_factory=remoteclip_factory,
    )
    remoteclip_test[caption_col] = {
        'class_ious': class_ious, 'miou': miou, 'oa': oa, 'loss': loss,
    }
    print(f'  test mIoU: {miou:.4f}, OA: {oa:.4f}')

In [ ]:
# Paired comparison: CLIP (Phase 2) vs RemoteCLIP (Phase 3), per caption
phase2_clip = multimodal_data['multimodal']

print(f"{'Caption':<22} {'CLIP':>10} {'RemoteCLIP':>12} {'Δ':>10} {'Rel %':>8}")
print('-' * 64)

for caption_col in CAPTION_COLS:
    clip_miou = phase2_clip[caption_col]['test_miou']
    rc_miou   = remoteclip_test[caption_col]['miou']
    delta = rc_miou - clip_miou
    rel = (delta / clip_miou) * 100
    print(f'{caption_col:<22} {clip_miou:>10.4f} {rc_miou:>12.4f} {delta:>+10.4f} {rel:>+7.2f}%')

# Best RemoteCLIP caption
best_rc_caption = max(remoteclip_test, key=lambda k: remoteclip_test[k]['miou'])
best_rc_miou = remoteclip_test[best_rc_caption]['miou']
print(f'\nBest RemoteCLIP caption: {best_rc_caption} (test mIoU = {best_rc_miou:.4f})')
print(f'Phase 2 best CLIP caption: {BEST_CAPTION} (test mIoU = {PHASE2_MIOU:.4f})')

In [ ]:
# Per-class IoU comparison on the best caption (text_qwen3-4b)
clip_ious = phase2_clip[BEST_CAPTION]['class_ious']
rc_ious   = remoteclip_test[BEST_CAPTION]['class_ious']

print(f'Per-class comparison on {BEST_CAPTION}:\n')
print(f"{'Class':<12} {'CLIP':>10} {'RemoteCLIP':>12} {'Δ':>10} {'Rel %':>8}")
print('-' * 54)

for i, name in enumerate(CLASS_NAMES):
    c = clip_ious[i]
    r = rc_ious[i]
    if c is None or (isinstance(r, float) and np.isnan(r)):
        print(f'{name:<12} {"N/A":>10} {"N/A":>12} {"":>10} {"":>8}')
        continue
    delta = r - c
    rel = (delta / c) * 100 if c > 0 else float('nan')
    print(f'{name:<12} {c:>10.4f} {r:>12.4f} {delta:>+10.4f} {rel:>+7.1f}%')

print('-' * 54)
clip_miou_best = phase2_clip[BEST_CAPTION]['test_miou']
rc_miou_best   = remoteclip_test[BEST_CAPTION]['miou']
print(f'{"mIoU":<12} {clip_miou_best:>10.4f} {rc_miou_best:>12.4f} '
      f'{rc_miou_best - clip_miou_best:>+10.4f} '
      f'{(rc_miou_best - clip_miou_best) / clip_miou_best * 100:>+7.1f}%')

In [ ]:
# Persist Ablation 1 results
ablation1_results = {
    'experiment': 'text_encoder_pretraining',
    'phase2_clip_test': {
        cap: {'miou': phase2_clip[cap]['test_miou'],
              'oa':   phase2_clip[cap]['test_oa'],
              'class_ious': phase2_clip[cap]['class_ious']}
        for cap in CAPTION_COLS
    },
    'phase3_remoteclip_test': {
        cap: {'miou': res['miou'], 'oa': res['oa'],
              'class_ious': [float(x) if not np.isnan(x) else None for x in res['class_ious']]}
        for cap, res in remoteclip_test.items()
    },
    'best_remoteclip_caption': best_rc_caption,
}

with open(RESULTS_DIR / 'ablation1_text_encoder.json', 'w') as f:
    json.dump(ablation1_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'ablation1_text_encoder.json'}")